# Thực nghiệm mô hình gợi ý dựa trên Đồ thị Tri thức (CKAN) trên 3 tập dữ liệu (Movie, Book, Music)
So sánh 4 mô hình: MostPopular, Matrix Factorization (MF), RippleNet (CIKM 2018), CKAN (SIGIR 2020)

Tài liệu tham khảo:
- CKAN: [Ze Wang et al., ACM SIGIR 2020](https://github.com/weberrr/CKAN)
- RippleNet: [Hongwei Wang et al., ACM CIKM 2018](https://github.com/hwwang55/RippleNet)

---

### Cấu trúc thực nghiệm:
1. **Phân chia dữ liệu**:
   - Tỷ lệ Train (60%) - Validation (20%) - Test (20%).
   - Đánh giá trên tập Validation qua từng epoch để lưu trạng thái tốt nhất (`best_val_auc`), sau đó đánh giá độc lập trên tập Test.
2. **Các mô hình so sánh**:
   - `MostPopular`: Baseline heuristic dựa trên tần suất tương tác.
   - `Matrix Factorization`: Lọc cộng tác ma trận nhân tố ẩn (BCE Loss).
   - `RippleNet`: Mô hình lan truyền gợn sóng sở thích người dùng trên đồ thị tri thức.
   - `CKAN`: Mô hình lan truyền hai chiều kết hợp mạng chú ý đồ thị tri thức hợp tác.
3. **Thang đo đánh giá**:
   - **Dự đoán CTR**: `ROC-AUC`, `F1-Score`, `Accuracy`.
   - **Gợi ý Top-K**: `Recall@K` với $K \in \{5, 10, 20, 50, 100\}$ trên 100 người dùng mẫu.
   - **Đánh giá độ thưa (Sparsity Analysis)**: Đo lường ROC-AUC qua 6 tỷ lệ dữ liệu huấn luyện: $10\%, 20\%, 40\%, 60\%, 80\%, 100\%$.


In [ ]:
import torch

print("PyTorch:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", device)
if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))



In [ ]:
!pip install -q --upgrade scikit-learn scipy matplotlib seaborn pandas tqdm

import os
import time
import json
import pickle
import copy
import shutil
import random
import zipfile
import urllib.request
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#D1D5DB'
plt.rcParams['axes.linewidth'] = 1.0

# Tạo thư mục lưu trữ model
os.makedirs("./saved_models", exist_ok=True)



## 2. Kiến trúc mô hình CKAN

### 2.1. Ý tưởng chính
Lọc cộng tác truyền thống (Matrix Factorization) tính điểm dựa trên tích vô hướng:
$$y_{u, v} = \sigma(\mathbf{u}_u^T \mathbf{v}_v)$$
Khi ma trận quá thưa hoặc gặp user/item mới (cold-start), mô hình không có đủ tương tác để học tốt.

CKAN kết hợp thêm Knowledge Graph (KG) theo hai nhánh:
- **User Branch**: Lan truyền sở thích từ các item người dùng từng tương tác sang các entity liên quan trên KG (đạo diễn, diễn viên, thể loại...).
- **Item Branch**: Lan truyền ngữ nghĩa từ item ứng viên sang các láng giềng trên KG.
- **Attention Layer**: Mạng MLP 3 tầng + Sigmoid + Softmax tính trọng số giữa quan hệ và ngữ cảnh.

```
                    [ Lịch sử tương tác User ]                      [ Item ứng viên: v ]
                                 │                                             │
                   0-hop: e_u^(0) = mean(e_items)                 0-hop: e_v^(0) = e_v
                                 │                                             │
                                 ▼                                             ▼
                    [ User Ripple Set: E_u^l ]                    [ Item Ripple Set: E_v^l ]
                          (h, r, t)                                     (h, r, t)
                                 │                                             │
                                 ▼                                             ▼
              ┌────────────────────────────────────┐         ┌────────────────────────────────────┐
              │          ATTENTION LAYER           │         │          ATTENTION LAYER           │
              │   s = MLP([e_h ; e_r])             │         │   s = MLP([e_h ; e_r])             │
              │   alpha = Softmax(Sigmoid(s))      │         │   alpha = Softmax(Sigmoid(s))      │
              │   e_u^l = sum(alpha * e_t)         │         │   e_v^l = sum(alpha * e_t)         │
              └──────────────────┬─────────────────┘         └──────────────────┬─────────────────┘
                                 │                                             │
                                 ▼                                             ▼
                        [ Concat Aggregator ]                         [ Concat Aggregator ]
                     e_u = [e_u^L; ...; e_u^0]                     e_v = [e_v^L; ...; e_v^0]
                                 └───────────────────────┬─────────────────────┘
                                                         │
                                                         ▼
                                                [ Điểm dự đoán ]
                                           y_hat = Sigmoid(e_u^T * e_v)
```



## 3. Cấu hình tham số thực nghiệm

Bảng tham số thực nghiệm:
- `dim = 64`: Kích thước embedding vector cho entity và relation.
- `l2_weight = 1e-5`: Hệ số chuẩn hóa L2 weight decay.
- `agg = 'concat'`: Phép tổng hợp nối vector các tầng lan truyền.
- `item_triple_set_size = 64`: Kích thước tập bộ ba láng giềng lấy mẫu của Item.
- `user_triple_set_size`: Kích thước tập bộ ba lấy mẫu của User (32 cho Movie, 16 cho Book, 8 cho Music).
- `n_layer`: Số tầng lan truyền tri thức (1 cho Movie, 2 cho Book và Music).


In [ ]:
CONFIG = {
    "movie": {
        "dim": 64,
        "n_layer": 1,
        "itss": 64,          # item_triple_set_size = 64
        "utss": 32,          # user_triple_set_size = 32 (theo repo gốc weberrr/CKAN)
        "lr": 0.002,         # learning rate mặc định repo gốc
        "l2_weight": 1e-5,   # l2 weight decay mặc định
        "batch_size": 2048,  # batch size theo repo gốc
        "n_epochs": 10,
        "agg": "concat"
    },
    "book": {
        "dim": 64,
        "n_layer": 2,        # repo gốc: --n_layer 2
        "itss": 64,          # item_triple_set_size = 64
        "utss": 16,          # user_triple_set_size = 16 (theo repo gốc weberrr/CKAN)
        "lr": 0.002,         # learning rate 0.002
        "l2_weight": 1e-5,
        "batch_size": 1024,
        "n_epochs": 8,
        "agg": "concat"
    },
    "music": {
        "dim": 64,
        "n_layer": 2,        # repo gốc: 2-hop cho Last.FM
        "itss": 64,          # item_triple_set_size = 64
        "utss": 8,           # user_triple_set_size = 8 (theo repo gốc weberrr/CKAN)
        "lr": 0.002,
        "l2_weight": 1e-5,
        "batch_size": 1024,
        "n_epochs": 10,
        "agg": "concat"
    }
}

active_datasets = ["movie", "book", "music"]


## 4. Dữ liệu thực nghiệm

Dữ liệu được nạp và lưu trữ tại thư mục `./data/`.


In [ ]:
# ============================================================
# Tải dữ liệu tương tác và đồ thị tri thức
# ============================================================
import os
import shutil
import urllib.request
import zipfile

DATASETS = ["movie", "book", "music"]
WEBERRR_RAW = "https://raw.githubusercontent.com/weberrr/CKAN/master/data"

# 1. Tải kg.txt và item_index2entity_id.txt từ repo gốc weberrr/CKAN
for ds in DATASETS:
    os.makedirs(f"./data/{ds}", exist_ok=True)
    for fname in ["item_index2entity_id.txt", "kg.txt"]:
        dest = f"./data/{ds}/{fname}"
        if not os.path.exists(dest) or os.path.getsize(dest) == 0:
            url = f"{WEBERRR_RAW}/{ds}/{fname}"
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=60) as resp, open(dest, "wb") as f:
                f.write(resp.read())

# 2. Tải raw ratings theo trích dẫn từ bài báo CKAN (RippleNet & KGCN)
RAW_SOURCES = {
    "music": {
        "file": "user_artists.dat",
        "url": "https://raw.githubusercontent.com/hwwang55/KGCN/master/data/music/user_artists.dat",
        "is_zip": False
    },
    "book": {
        "file": "BX-Book-Ratings.csv",
        "url": "https://raw.githubusercontent.com/hwwang55/RippleNet/master/data/book/BX-Book-Ratings.csv",
        "is_zip": False
    },
    "movie": {
        "file": "ratings.csv",
        "url": "http://files.grouplens.org/datasets/movielens/ml-20m.zip",
        "is_zip": True
    }
}

for ds in DATASETS:
    if os.path.exists(f"./data/{ds}/ratings_final.npy") and os.path.exists(f"./data/{ds}/kg_final.npy"):
        continue

    cfg = RAW_SOURCES[ds]
    dest = f"./data/{ds}/{cfg['file']}"
    if not os.path.exists(dest):
        try:
            if cfg["is_zip"]:
                zip_tmp = f"./data/{ds}/temp.zip"
                urllib.request.urlretrieve(cfg["url"], zip_tmp)
                with zipfile.ZipFile(zip_tmp, "r") as z:
                    for name in z.namelist():
                        if name.endswith("ratings.csv"):
                            with z.open(name) as src, open(dest, "wb") as dst:
                                shutil.copyfileobj(src, dst)
                            break
                if os.path.exists(zip_tmp):
                    os.remove(zip_tmp)
            else:
                req = urllib.request.Request(cfg["url"], headers={"User-Agent": "Mozilla/5.0"})
                with urllib.request.urlopen(req, timeout=60) as resp, open(dest, "wb") as f:
                    f.write(resp.read())
        except Exception:
            pass

print("Dữ liệu đã sẵn sàng trong ./data/")


In [ ]:
# ============================================================
# Tiền xử lý dữ liệu tương tác và đồ thị tri thức
# ============================================================
RATING_FILE_NAME = {
    "music": "user_artists.dat",
    "book": "BX-Book-Ratings.csv",
    "movie": "ratings.csv"
}
SEP = {"music": "\t", "book": ";", "movie": ","}
THRESHOLD = {"music": 0, "book": 0, "movie": 4}

def read_item_index_to_entity_id_file(dataset):
    file = f"./data/{dataset}/item_index2entity_id.txt"
    item_index_old2new = {}
    entity_id2index = {}
    i = 0
    with open(file, encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) >= 2:
                item_index, satori_id = parts[0], parts[1]
                item_index_old2new[item_index] = i
                entity_id2index[satori_id] = i
                i += 1
    return item_index_old2new, entity_id2index

def convert_rating(dataset, item_index_old2new):
    file = f"./data/{dataset}/{RATING_FILE_NAME[dataset]}"
    if not os.path.exists(file):
        return

    item_set = set(item_index_old2new.values())
    user_pos_ratings = defaultdict(set)
    user_neg_ratings = defaultdict(set)

    with open(file, encoding="utf-8", errors="ignore") as f:
        _ = f.readline()
        for line in f:
            array = line.strip().split(SEP[dataset])
            if dataset == "book":
                array = [x.strip('"') for x in array]
            if len(array) < 3:
                continue
            item_index_old = array[1]
            if item_index_old not in item_index_old2new:
                continue
            item_index = item_index_old2new[item_index_old]
            user_index_old = array[0]
            try:
                rating = float(array[2])
            except ValueError:
                continue

            if rating >= THRESHOLD[dataset]:
                user_pos_ratings[user_index_old].add(item_index)
            else:
                user_neg_ratings[user_index_old].add(item_index)

    # Lọc tương tác positive theo ngưỡng; Movie lấy mẫu 2,500 users để tối ưu hóa bộ nhớ
    target_users = user_pos_ratings
    if dataset == "movie" and len(target_users) > 2500:
        np.random.seed(42)
        sample_keys = np.random.choice(list(target_users.keys()), size=2500, replace=False)
        target_users = {u: target_users[u] for u in sample_keys}

    write_file = f"./data/{dataset}/ratings_final.txt"
    with open(write_file, "w", encoding="utf-8") as writer:
        user_cnt = 0
        user_index_old2new = {}
        for user_index_old, pos_item_set in target_users.items():
            if user_index_old not in user_index_old2new:
                user_index_old2new[user_index_old] = user_cnt
                user_cnt += 1
            user_index = user_index_old2new[user_index_old]
            for item in pos_item_set:
                writer.write(f"{user_index}\t{item}\t1\n")
            unwatched_set = item_set - pos_item_set
            if user_index_old in user_neg_ratings:
                unwatched_set -= user_neg_ratings[user_index_old]
            unwatched_list = list(unwatched_set)
            if len(unwatched_list) >= len(pos_item_set):
                neg_items = np.random.choice(unwatched_list, size=len(pos_item_set), replace=False)
                for item in neg_items:
                    writer.write(f"{user_index}\t{item}\t0\n")

    r_np = np.loadtxt(write_file, dtype=np.int32)
    np.save(f"./data/{dataset}/ratings_final.npy", r_np)

def convert_kg(dataset, entity_id2index):
    file = f"./data/{dataset}/kg.txt"
    if not os.path.exists(file):
        return

    write_file = f"./data/{dataset}/kg_final.txt"
    entity_cnt = len(entity_id2index)
    relation_id2index = {}
    relation_cnt = 0

    with open(file, encoding="utf-8") as f, open(write_file, "w", encoding="utf-8") as writer:
        for line in f:
            array = line.strip().split("\t")
            if len(array) != 3:
                continue
            head_old, relation_old, tail_old = array[0], array[1], array[2]
            if head_old not in entity_id2index:
                entity_id2index[head_old] = entity_cnt
                entity_cnt += 1
            head = entity_id2index[head_old]
            if tail_old not in entity_id2index:
                entity_id2index[tail_old] = entity_cnt
                entity_cnt += 1
            tail = entity_id2index[tail_old]
            if relation_old not in relation_id2index:
                relation_id2index[relation_old] = relation_cnt
                relation_cnt += 1
            relation = relation_id2index[relation_old]
            writer.write(f"{head}\t{relation}\t{tail}\n")

    k_np = np.loadtxt(write_file, dtype=np.int32)
    np.save(f"./data/{dataset}/kg_final.npy", k_np)

def preprocess_and_cache(ds_name, force_rebuild=False):
    r_npy = f"./data/{ds_name}/ratings_final.npy"
    k_npy = f"./data/{ds_name}/kg_final.npy"
    if not force_rebuild and os.path.exists(r_npy) and os.path.exists(k_npy) and os.path.getsize(r_npy) > 0:
        return
    item_index_old2new, entity_id2index = read_item_index_to_entity_id_file(ds_name)
    convert_rating(ds_name, item_index_old2new)
    convert_kg(ds_name, entity_id2index)

for ds in active_datasets:
    # Nếu bộ movie có > 500k tương tác thì rebuild lại về đúng mẫu benchmark
    r_check = f"./data/{ds}/ratings_final.npy"
    force = False
    if os.path.exists(r_check):
        r_temp = np.load(r_check)
        if len(r_temp) > 500000:
            force = True
        # Nếu book bị mất cold-start (< 10% do bị lọc nhầm trước đó), rebuild lại đúng chuẩn repo gốc
        if ds == "book":
            u_c = Counter(r_temp[:, 0])
            c_pct = sum(1 for c in u_c.values() if c <= 5) / len(u_c) * 100
            if c_pct < 10.0:
                force = True
    preprocess_and_cache(ds, force_rebuild=force)
    r_len = len(np.load(f"./data/{ds}/ratings_final.npy")) if os.path.exists(f"./data/{ds}/ratings_final.npy") else 0
    k_len = len(np.load(f"./data/{ds}/kg_final.npy")) if os.path.exists(f"./data/{ds}/kg_final.npy") else 0
    print(f"{ds}: ratings={r_len:,} | kg_triples={k_len:,}")


### 4.2. Cấu trúc dữ liệu sau tiền xử lý

Sau bước tiền xử lý, mỗi miền dữ liệu (`movie`, `book`, `music`) được chuẩn hóa thành 3 file chính để nạp vào mô hình:

1. **`ratings_final.npy`**: Ma trận tương tác người dùng - sản phẩm `[User_ID, Item_ID, Label]`.
   - `User_ID`: Mã định danh người dùng ($0 \dots N_{user}-1$).
   - `Item_ID`: Mã sản phẩm ($0 \dots N_{item}-1$).
   - `Label`: Nhãn nhị phân ($1$ = Tương tác tích cực, $0$ = Tương tác âm được lấy mẫu 1:1).
2. **`kg_final.npy`**: Đồ thị tri thức `[Head_Entity, Relation_ID, Tail_Entity]`.
   - `Head / Tail`: Thực thể đầu và đuôi. Các Item nằm ở dải $[0, N_{item}-1]$, các thực thể ngữ nghĩa mở rộng (đạo diễn, tác giả, thể loại...) nằm ở dải $[N_{item}, N_{entity}-1]$.
   - `Relation_ID`: Mã định danh quan hệ tri thức ($0 \dots N_{relation}-1$).
3. **`item_index2entity_id.txt`**: Bảng ánh xạ từ mã ID gốc của hệ thống (MovieID, ISBN sách, ArtistID) sang ID nội bộ của mô hình.


In [ ]:
# ============================================================
# KIỂM TRA MẪU DỮ LIỆU SAU TIỀN XỬ LÝ (SAMPLE & SCHEMA)
# ============================================================
summary_records = []

for ds in active_datasets:
    r = np.load(f"./data/{ds}/ratings_final.npy")
    k = np.load(f"./data/{ds}/kg_final.npy")
    
    # Đọc 3 dòng mapping đầu tiên
    map_file = f"./data/{ds}/item_index2entity_id.txt"
    sample_maps = []
    if os.path.exists(map_file):
        with open(map_file, "r", encoding="utf-8") as f:
            for _ in range(3):
                line = f.readline()
                if line: sample_maps.append(line.strip().split("\t"))
                
    n_u = int(r[:, 0].max()) + 1
    n_i = max(int(r[:, 1].max()), int(k[:, 0].max())) + 1
    n_e = max(int(r[:, 1].max()), int(k[:, 0].max()), int(k[:, 2].max())) + 1
    n_rel = int(k[:, 1].max()) + 1
    
    summary_records.append({
        "Dataset": ds.capitalize(),
        "Ratings_Shape": str(r.shape),
        "KG_Shape": str(k.shape),
        "Users [0..max]": f"0 .. {n_u - 1}",
        "Items [0..max]": f"0 .. {n_i - 1}",
        "Entities [0..max]": f"0 .. {n_e - 1}",
        "Relations [0..max]": f"0 .. {n_rel - 1}",
        "Dtype": str(r.dtype),
        "RAM (MB)": f"{(r.nbytes + k.nbytes) / (1024*1024):.2f}"
    })

print("--- TỔNG QUAN ĐỊNH DẠNG DỮ LIỆU ---")
print(pd.DataFrame(summary_records).to_string(index=False))

print("\n--- MẪU DỮ LIỆU CHI TIẾT THEO TỪNG MIỀN ---")
for ds in active_datasets:
    r = np.load(f"./data/{ds}/ratings_final.npy")
    k = np.load(f"./data/{ds}/kg_final.npy")
    
    df_r_sample = pd.DataFrame(r[:3], columns=["User_ID", "Item_ID", "Label (1/0)"])
    df_k_sample = pd.DataFrame(k[:3], columns=["Head_Entity", "Relation_ID", "Tail_Entity"])
    
    print(f"\n[{ds.upper()}]")
    print("  • 3 dòng đầu ratings_final.npy:")
    print(df_r_sample.to_string(index=False))
    print("  • 3 dòng đầu kg_final.npy:")
    print(df_k_sample.to_string(index=False))
    
    # In mẫu mapping
    map_file = f"./data/{ds}/item_index2entity_id.txt"
    if os.path.exists(map_file):
        with open(map_file, "r", encoding="utf-8") as f:
            lines = [f.readline().strip().split("\t") for _ in range(2)]
            print(f"  • Mẫu mapping (ID gốc -> Entity ID): {lines[0]} | {lines[1]}")


## 5. Khám phá dữ liệu (EDA)

Phân tích các đặc trưng chính của 3 tập dữ liệu:
1. **Độ thưa (Sparsity)**: Tỉ lệ ô trống trong ma trận tương tác.
2. **Phân bố tương tác (Long-tail)**: Nhóm item phổ biến chiếm bao nhiêu % lượng tương tác.
3. **User Activity**: Phân bố mức độ tương tác và tỉ lệ user cold-start ($\le 5$ tương tác).
4. **Cấu trúc KG**: Số lượng entity, quan hệ, triple và phân bố bậc kết nối.



In [ ]:
eda_summary = []

for ds in active_datasets:
    r_data = np.load(f"./data/{ds}/ratings_final.npy")
    k_data = np.load(f"./data/{ds}/kg_final.npy")
    
    n_users = int(r_data[:, 0].max()) + 1
    n_items = max(int(r_data[:, 1].max()), int(k_data[:, 0].max())) + 1
    n_ratings = len(r_data)
    
    sparsity = (1.0 - n_ratings / (n_users * n_items)) * 100
    
    user_counts = list(Counter(r_data[:, 0]).values())
    avg_u_inter = float(np.mean(user_counts))
    median_u_inter = float(np.median(user_counts))
    cold_start_users = sum(1 for c in user_counts if c <= 5) / n_users * 100
    
    item_counts = sorted(list(Counter(r_data[:, 1]).values()), reverse=True)
    cum_inter = np.cumsum(item_counts) / n_ratings * 100
    top20_share = cum_inter[min(int(0.2 * len(item_counts)), len(cum_inter) - 1)]
    
    n_entities = max(int(r_data[:, 1].max()), int(k_data[:, 0].max()), int(k_data[:, 2].max())) + 1
    n_relations = int(k_data[:, 1].max()) + 1
    n_triples = len(k_data)
    branching_factor = n_triples / n_entities
    
    eda_summary.append({
        "Dataset": ds.capitalize(),
        "Users": n_users,
        "Items": n_items,
        "Ratings": n_ratings,
        "Sparsity (%)": round(sparsity, 2),
        "Median_Inter": round(median_u_inter, 1),
        "Avg_Inter/User": round(avg_u_inter, 1),
        "ColdStart_Users (%)": round(cold_start_users, 1),
        "Top20%_Item_Share (%)": round(top20_share, 1),
        "KG_Entities": n_entities,
        "KG_Relations": n_relations,
        "KG_Triples": n_triples,
        "KG_Branching": round(branching_factor, 2)
    })

df_eda = pd.DataFrame(eda_summary)
print(df_eda.to_string(index=False))


In [ ]:
# ============================================================
# HÌNH 1: KHÁM PHÁ PHÂN PHỐI DỮ LIỆU TƯƠNG TÁC (3 DATASETS)
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(15, 11), dpi=300)

labels = [ds.capitalize() for ds in active_datasets]
x = np.arange(len(labels))
w = 0.25

users = [df_eda.loc[i, "Users"] for i in range(len(active_datasets))]
items = [df_eda.loc[i, "Items"] for i in range(len(active_datasets))]
ratings = [df_eda.loc[i, "Ratings"] for i in range(len(active_datasets))]

# 1. Quy mô tương tác người dùng (Grouped Bar Chart - Thang Log)
b1 = axes[0, 0].bar(x - w, users, w, label="Users", color="#3B82F6")
b2 = axes[0, 0].bar(x, items, w, label="Items", color="#10B981")
b3 = axes[0, 0].bar(x + w, ratings, w, label="Ratings", color="#F59E0B")
axes[0, 0].set_yscale("log")
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(labels, fontweight="bold", fontsize=11)
axes[0, 0].set_title("1. Quy mô User, Item, Rating (Thang Log)", fontweight="bold", fontsize=12)
axes[0, 0].set_ylabel("Số lượng (Log Scale)")
axes[0, 0].legend(loc="upper right")
axes[0, 0].grid(axis="y", linestyle=":", alpha=0.6)

for grp in [b1, b2, b3]:
    for bar in grp:
        h = bar.get_height()
        axes[0, 0].text(bar.get_x() + bar.get_width()/2, h * 1.15, f"{int(h):,}",
                        ha="center", va="bottom", fontsize=8, rotation=25)

# 2. Độ thưa & Tỉ lệ Cold-Start Users (Bar Chart)
sparsities = [df_eda.loc[i, "Sparsity (%)"] for i in range(len(active_datasets))]
cold_starts = [df_eda.loc[i, "ColdStart_Users (%)"] for i in range(len(active_datasets))]
w2 = 0.35

b_sp = axes[0, 1].bar(x - w2/2, sparsities, w2, label="Độ thưa ma trận (%)", color="#EF4444")
b_cs = axes[0, 1].bar(x + w2/2, cold_starts, w2, label="User Cold-Start (<=5 rating %)", color="#8B5CF6")
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(labels, fontweight="bold", fontsize=11)
axes[0, 1].set_title("2. Độ thưa ma trận & Tỉ lệ Cold-Start Users (%)", fontweight="bold", fontsize=12)
axes[0, 1].set_ylabel("Tỉ lệ (%)")
axes[0, 1].set_ylim(0, 115)
axes[0, 1].legend(loc="upper right")
axes[0, 1].grid(axis="y", linestyle=":", alpha=0.6)

for bar in b_sp:
    axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5, f"{bar.get_height():.2f}%",
                    ha="center", va="bottom", fontsize=9, fontweight="bold")
for bar in b_cs:
    axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5, f"{bar.get_height():.1f}%",
                    ha="center", va="bottom", fontsize=9, fontweight="bold")

# 3. Phân phối tương tác trên mỗi User (Boxplot)
user_counts_all = [list(Counter(np.load(f"./data/{ds}/ratings_final.npy")[:, 0]).values()) for ds in active_datasets]
bp = axes[1, 0].boxplot(user_counts_all, tick_labels=labels, patch_artist=True,
                         boxprops=dict(facecolor="#E0E7FF", color="#4338CA"),
                         medianprops=dict(color="#DC2626", linewidth=2.5),
                         whiskerprops=dict(color="#4338CA"),
                         capprops=dict(color="#4338CA"),
                         flierprops=dict(marker="o", markersize=3, alpha=0.3))
axes[1, 0].set_yscale("log")
axes[1, 0].set_title("3. Phân phối tương tác / User (Boxplot, đường đỏ: Median)", fontweight="bold", fontsize=12)
axes[1, 0].set_ylabel("Số tương tác / user (Log Scale)")
axes[1, 0].grid(axis="y", linestyle=":", alpha=0.6)

# 4. Hiện tượng Đuôi dài Long-Tail (Pareto 80/20 Lorenz Curve)
colors = {"movie": "#F59E0B", "book": "#10B981", "music": "#3B82F6"}
for ds in active_datasets:
    r_arr = np.load(f"./data/{ds}/ratings_final.npy")
    item_counts = sorted(list(Counter(r_arr[:, 1]).values()), reverse=True)
    cum_pct = np.cumsum(item_counts) / np.sum(item_counts) * 100
    x_pct = np.linspace(0, 100, len(cum_pct))
    axes[1, 1].plot(x_pct, cum_pct, label=f"{ds.capitalize()}", color=colors.get(ds, "#6B7280"), lw=2.2)

axes[1, 1].axvline(20, color="#DC2626", linestyle="--", lw=1.5, label="Mốc 20% Items")
axes[1, 1].axhline(80, color="#6B7280", linestyle=":", lw=1.2, label="Mốc 80% Tương tác")
axes[1, 1].set_title("4. Hiện tượng Đuôi dài Long-Tail (Đường cong Pareto 80/20)", fontweight="bold", fontsize=12)
axes[1, 1].set_xlabel("% Sản phẩm (xếp từ nhiều đến ít tương tác)")
axes[1, 1].set_ylabel("% Tương tác tích lũy")
axes[1, 1].set_xlim(0, 100)
axes[1, 1].set_ylim(0, 105)
axes[1, 1].legend(loc="lower right")
axes[1, 1].grid(True, linestyle=":", alpha=0.6)

plt.suptitle("Đặc trưng phân phối dữ liệu tương tác (MovieLens vs Book-Crossing vs Last.FM)", fontsize=14, fontweight="bold", y=0.99)
plt.tight_layout()
plt.show()

# ============================================================
# HÌNH 2: ĐẶC TRƯNG ĐỒ THỊ TRI THỨC KNOWLEDGE GRAPH
# ============================================================
fig2, axes2 = plt.subplots(1, 2, figsize=(15, 5.5), dpi=300)

kg_triples = [df_eda.loc[i, "KG_Triples"] for i in range(len(active_datasets))]
kg_entities = [df_eda.loc[i, "KG_Entities"] for i in range(len(active_datasets))]

# 1. Quy mô Knowledge Graph
b_tr = axes2[0].bar(x - w2/2, kg_triples, w2, label="Triples (Cạnh tri thức)", color="#059669")
b_en = axes2[0].bar(x + w2/2, kg_entities, w2, label="Entities (Thực thể)", color="#0D9488")
axes2[0].set_yscale("log")
axes2[0].set_xticks(x)
axes2[0].set_xticklabels(labels, fontweight="bold", fontsize=11)
axes2[0].set_title("Quy mô Knowledge Graph: Triples & Entities (Thang Log)", fontweight="bold", fontsize=12)
axes2[0].set_ylabel("Số lượng (Log Scale)")
axes2[0].legend()
axes2[0].grid(axis="y", linestyle=":", alpha=0.6)

for grp in [b_tr, b_en]:
    for bar in grp:
        h = bar.get_height()
        axes2[0].text(bar.get_x() + bar.get_width()/2, h * 1.15, f"{int(h):,}",
                      ha="center", va="bottom", fontsize=9, fontweight="bold", rotation=15)

# 2. Hệ số phân nhánh KG (Branching factor)
branches = [df_eda.loc[i, "KG_Branching"] for i in range(len(active_datasets))]
b_br = axes2[1].bar(x, branches, 0.45, color="#6366F1", edgecolor="#4338CA")
axes2[1].set_xticks(x)
axes2[1].set_xticklabels(labels, fontweight="bold", fontsize=11)
axes2[1].set_title("Mật độ liên kết trung bình / Thực thể (Branching Factor = Triples / Entities)", fontweight="bold", fontsize=12)
axes2[1].set_ylabel("Số cạnh / thực thể")
axes2[1].grid(axis="y", linestyle=":", alpha=0.6)

for bar in b_br:
    axes2[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f"{bar.get_height():.2f}",
                  ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.suptitle("Đặc trưng Đồ thị tri thức (Knowledge Graph Structure)", fontsize=14, fontweight="bold", y=0.99)
plt.tight_layout()
plt.show()


## PHẦN A: CÁC MÔ HÌNH BASELINE (MostPopular, Matrix Factorization)

### 1. MostPopular (Baseline heuristic không cá nhân hóa)
- Tính tần suất xuất hiện của các item dương tính ($r=1$) trong tập huấn luyện. Điểm số của item là tỉ lệ xuất hiện chuẩn hóa:
  $$s(v) = \frac{\text{Count}(v)}{\max_{v'} \text{Count}(v')}$$
- Gợi ý danh sách giống nhau cho mọi người dùng. Phục vụ làm ngưỡng sàn (baseline sàn) để đánh giá giá trị của việc cá nhân hóa.

### 2. Matrix Factorization (Latent Factor Model / BPR-MF)
- Lọc cộng tác cổ điển biểu diễn người dùng và sản phẩm trong không gian vector ẩn $d$ chiều:
  $$\hat{y}_{uv} = \sigma\left( \mathbf{u}_u^T \mathbf{v}_v + b_u + b_v \right)$$
- Trong đó $\mathbf{u}_u, \mathbf{v}_v \in \mathbb{R}^d$ là embedding vector của user $u$ và item $v$; $b_u, b_v$ là hệ số chệch (bias).
- Tối ưu hóa bằng Binary Cross-Entropy Loss (BCE) kết hợp Adam Optimizer và chuẩn hóa L2 weight decay.
- **Hạn chế**: Chỉ dựa vào ma trận tương tác User-Item. Khi dữ liệu thưa thớt (>99%) hoặc gặp user/item ít tương tác, mô hình không có đủ tín hiệu để cập nhật embedding.


In [ ]:
# ============================================================
# Các mô hình baseline (MostPopular & Matrix Factorization)
# ============================================================

# --- 1. MOST POPULAR ---
class MostPopularBaseline:
    """
    Gợi ý dựa trên độ phổ biến: Xếp hạng item theo số lượt tương tác positive trong train set.
    """
    def __init__(self):
        self.item_scores = defaultdict(float)

    def fit(self, train_data, n_items):
        pos = train_data[train_data[:, 2] == 1]
        counts = Counter(pos[:, 1])
        max_c = max(counts.values()) if counts else 1.0
        for i in range(n_items):
            self.item_scores[i] = counts.get(i, 0) / max_c

    def predict(self, users, items):
        return np.array([self.item_scores.get(int(i), 0.0) for i in items])

    def score_all_items(self, u, n_items):
        return np.array([self.item_scores.get(i, 0.0) for i in range(n_items)])


# --- 2. MATRIX FACTORIZATION (Latent Factor Model / BPR-MF) ---
class MatrixFactorizationBaseline(nn.Module):
    """
    Matrix Factorization với User/Item Embedding và Bias terms.
    Tối ưu hóa bằng BCE Loss trên tương tác nhị phân implicit feedback.
    """
    def __init__(self, n_user, n_item, dim=64):
        super().__init__()
        self.user_emb = nn.Embedding(n_user, dim)
        self.item_emb = nn.Embedding(n_item, dim)
        self.user_bias = nn.Embedding(n_user, 1)
        self.item_bias = nn.Embedding(n_item, 1)
        
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, users, items):
        # dot product: [batch_size, 1]
        dot = (self.user_emb(users) * self.item_emb(items)).sum(dim=-1, keepdim=True)
        logits = dot + self.user_bias(users) + self.item_bias(items)
        return torch.sigmoid(logits.squeeze(-1))

    def score_all_items(self, u, device):
        """
        Tính điểm số cho tất cả item đối với user u qua phép nhân ma trận nhanh.
        """
        u_t = torch.LongTensor([u]).to(device)
        u_e = self.user_emb(u_t) # [1, dim]
        u_b = self.user_bias(u_t).squeeze(-1) # [1]
        dots = torch.matmul(u_e, self.item_emb.weight.T).squeeze(0) + u_b + self.item_bias.weight.squeeze(-1)
        return torch.sigmoid(dots).detach().cpu().numpy()


## PHẦN B: MÔ HÌNH RIPPLENET (CIKM 2018) — CƠ CHẾ LAN TRUYỀN GỢN SÓNG TRI THỨC

Bài báo: **RippleNet: Propagating User Preferences on the Knowledge Graph for Recommender Systems**  
Tác giả: *Hongwei Wang, Fuzheng Zhang, Jialin Wang, Miao Zhao, Wenjie Li, Xing Xie, Minyi Guo* (ACM CIKM 2018)  
Tham khảo: [hwwang55/RippleNet](https://github.com/hwwang55/RippleNet)

---

### 1. Cơ chế lan truyền sở thích (Preference Ripple Propagation)
Sở thích người dùng xuất phát từ tập sản phẩm từng tương tác ($V_u$) và lan truyền qua các liên kết trên Đồ thị Tri thức (KG):
- **Tầng 0 (0-hop)**: Các sản phẩm người dùng đã tương tác trong quá khứ.
- **Tầng 1 (1-hop)**: Các thực thể liên kết trực tiếp (đạo diễn, diễn viên, tác giả, nghệ sĩ).
- **Tầng 2 (2-hop)**: Các thực thể liên quan bậc hai.

```
                    [ Lịch sử tương tác V_u ]
                               │
                               ▼ (hop 1)
                 ┌───────────────────────────────┐
                 │ Gợn sóng 1 (S_u^1): Thuộc tính trực tiếp │
                 └─────────────┬─────────────────┘
                               │
                               ▼ (hop 2)
                 ┌───────────────────────────────┐
                 │ Gợn sóng 2 (S_u^2): Thực thể liên quan  │
                 └───────────────────────────────┘
```

---

### 2. Định nghĩa Ripple Set
Tập thực thể khởi đầu ở tầng 0:
$$\mathcal{E}_u^0 = \mathcal{V}_u = \{v \mid y_{uv} = 1\}$$

Tập **gợn sóng tri thức (Ripple Set)** ở tầng thứ $k$ ($k = 1, 2, \dots, L$):
$$\mathcal{S}_u^k = \left\{ (h, r, t) \mid (h, r, t) \in \mathcal{G} \quad \text{và} \quad h \in \mathcal{E}_u^{k-1} \right\}$$
Tập thực thể đuôi tương ứng:
$$\mathcal{E}_u^k = \left\{ t \mid (h, r, t) \in \mathcal{S}_u^k \right\}$$

---

### 3. Cơ chế chú ý (Ripple Attention)
Với item ứng viên $v$ và từng bộ ba $(h_i, r_i, t_i) \in \mathcal{S}_u^k$:

1. **Trọng số chú ý $p_i$**:
   $$p_i = \text{Softmax}_i\left( \frac{\mathbf{v}^T (\mathbf{r}_i \odot \mathbf{h}_i)}{\sqrt{d}} \right)$$

2. **Phản hồi gợn sóng (Ripple Response $\mathbf{o}_u^k$)**:
   $$\mathbf{o}_u^k = \sum_{(h_i, r_i, t_i) \in \mathcal{S}_u^k} p_i \mathbf{t}_i$$

3. **Biểu diễn người dùng tổng hợp**:
   $$\mathbf{u} = \mathbf{u}_0 + \sum_{k=1}^L \mathbf{o}_u^k$$
   Trong đó $\mathbf{u}_0 = \frac{1}{|\mathcal{V}_u|} \sum_{v' \in \mathcal{V}_u} \mathbf{v}'$.

4. **Dự đoán xác suất click**:
   $$\hat{y}_{uv} = \sigma\left( \mathbf{u}^T \mathbf{v} \right)$$

---

### 4. So sánh RippleNet (CIKM 2018) và CKAN (SIGIR 2020)

| Tiêu chí | RippleNet (CIKM 2018) | CKAN (SIGIR 2020) |
|---|---|---|
| **Hướng lan truyền** | 1 chiều: Chỉ lan truyền từ User ($S_u^k$), Item là vector tĩnh $\mathbf{v}$. | 2 chiều: Đồng thời lan truyền cả User Ripple ($S_u^k$) và Item Ripple ($S_v^k$). |
| **Collaborative Propagation** | Không có. Chỉ duyệt KG từ các item đã tương tác. | Có: Mở rộng sang cả các người dùng đồng tương tác. |
| **Cơ chế Attention** | Bilinear tương quan tuyến tính: $\mathbf{v}^T (\mathbf{r} \odot \mathbf{h})$. | Mạng MLP 3 tầng phi tuyến: $s = \text{MLP}([\mathbf{h}; \mathbf{r}])$ + Sigmoid + Softmax. |
| **Biểu diễn Item** | Vector lookup tĩnh $\mathbf{v}$. | Vector động tổng hợp từ láng giềng KG: $\mathbf{e}_v = [\mathbf{e}_v^L; \dots; \mathbf{e}_v^0]$. |


In [ ]:
# ============================================================
# BỘ LẤY MẪU GỢN SÓNG TRI THỨC (Knowledge Ripple Sampler)
# Cung cấp Ripple Sets cho cả mô hình RippleNet và CKAN
# ============================================================
class KnowledgeRippleSampler:
    """
    Lấy mẫu láng giềng nhiều tầng (multi-hop ripple sets) trên Knowledge Graph.
    - Đối với Item: Duyệt từ node item sang các láng giềng trên KG.
    - Đối với User: Duyệt từ các item người dùng từng tương tác (0-hop) sang các thực thể liên quan.
    """
    def __init__(self, kg_np, n_layer=1, itss=64, utss=8):
        self.n_layer = n_layer
        self.itss = itss
        self.utss = utss
        
        self.kg_dict = defaultdict(list)
        for h, r, t in kg_np:
            self.kg_dict[int(h)].append((int(t), int(r)))

    def _empty_ripple(self, size):
        return [([0] * size, [0] * size, [0] * size) for _ in range(self.n_layer)]

    def build_item_ripple_set(self, n_items):
        item_triple_set = dict()
        for it in range(n_items):
            item_triple_set[it] = []
            for l in range(self.n_layer):
                h_list, r_list, t_list = [], [], []
                if l == 0:
                    entities = [it]
                else:
                    entities = item_triple_set[it][-1][2]
                
                for entity in entities:
                    for t, r in self.kg_dict.get(entity, []):
                        h_list.append(entity)
                        r_list.append(r)
                        t_list.append(t)
                        
                if len(h_list) == 0:
                    item_triple_set[it].append(([0] * self.itss, [0] * self.itss, [0] * self.itss))
                else:
                    replace = len(h_list) < self.itss
                    indices = np.random.choice(len(h_list), size=self.itss, replace=replace)
                    item_triple_set[it].append((
                        [h_list[i] for i in indices],
                        [r_list[i] for i in indices],
                        [t_list[i] for i in indices]
                    ))
        return item_triple_set

    def build_user_ripple_set(self, train_pos_dict, n_users=None):
        user_triple_set = dict()
        all_users = set(train_pos_dict.keys())
        if n_users is not None:
            all_users = all_users.union(range(n_users))
            
        for u in all_users:
            user_triple_set[u] = []
            hist = list(train_pos_dict.get(u, []))
            
            if not hist:
                user_triple_set[u] = self._empty_ripple(self.utss)
                continue
                
            for l in range(self.n_layer):
                h_list, r_list, t_list = [], [], []
                if l == 0:
                    entities = hist
                else:
                    entities = user_triple_set[u][-1][2]
                    
                for entity in entities:
                    for t, r in self.kg_dict.get(entity, []):
                        h_list.append(entity)
                        r_list.append(r)
                        t_list.append(t)
                        
                if len(h_list) == 0:
                    user_triple_set[u].append(([0] * self.utss, [0] * self.utss, [0] * self.utss))
                else:
                    replace = len(h_list) < self.utss
                    indices = np.random.choice(len(h_list), size=self.utss, replace=replace)
                    user_triple_set[u].append((
                        [h_list[i] for i in indices],
                        [r_list[i] for i in indices],
                        [t_list[i] for i in indices]
                    ))
        return user_triple_set


In [ ]:
# ============================================================
# Mô hình RippleNet
# ============================================================
class RippleNet(nn.Module):
    """
    Mô hình RippleNet (CIKM 2018): Lan truyền gợn sóng sở thích người dùng trên Knowledge Graph.
    - User representation tích lũy động từ các tầng ripple sets (h, r, t) dựa trên attention với Item candidate.
    - Item là vector embedding tĩnh v.
    - score_all_items được tối ưu hóa ma trận Vectorized để tính điểm nhanh toàn bộ danh mục item.
    """
    def __init__(self, n_entity, n_relation, dim=64, n_layer=1):
        super().__init__()
        self.n_entity = n_entity
        self.n_relation = n_relation
        self.dim = dim
        self.n_layer = n_layer

        self.entity_emb = nn.Embedding(n_entity, dim)
        self.relation_emb = nn.Embedding(n_relation, dim)
        
        nn.init.xavier_uniform_(self.entity_emb.weight)
        nn.init.xavier_uniform_(self.relation_emb.weight)

    def forward(self, items, user_triples):
        """
        items: Tensor [batch_size]
        user_triples: List of tuples (h, r, t), mỗi tuple gồm 3 LongTensor [batch_size, utss]
        """
        # [batch_size, dim]
        item_emb = self.entity_emb(items)
        
        # 0-hop: Vector trung bình của các thực thể đã tương tác ban đầu
        user_emb_0 = self.entity_emb(user_triples[0][0]).mean(dim=1) # [batch_size, dim]
        u_rep = user_emb_0
        v_exp = item_emb.unsqueeze(1) # [batch_size, 1, dim]
        
        for l in range(self.n_layer):
            h, r, t = user_triples[l]
            h_e = self.entity_emb(h) # [batch_size, utss, dim]
            r_e = self.relation_emb(r) # [batch_size, utss, dim]
            t_e = self.entity_emb(t) # [batch_size, utss, dim]
            
            # Biến đổi quan hệ R * h trong không gian vector (phép nhân từng phần tử - ma trận đường chéo)
            Rh = r_e * h_e # [batch_size, utss, dim]
            
            # Trọng số tương quan p_i = Softmax(v^T * Rh / sqrt(dim))
            logits = (v_exp * Rh).sum(dim=-1) / np.sqrt(self.dim) # [batch_size, utss]
            weights = F.softmax(logits, dim=-1).unsqueeze(-1) # [batch_size, utss, 1]
            
            # Phản hồi gợn sóng tầng l: o^l = sum(p_i * t_i)
            o_l = (weights * t_e).sum(dim=1) # [batch_size, dim]
            u_rep = u_rep + o_l
            
        # Dự đoán xác suất click: y_hat = Sigmoid(u^T * v)
        scores = torch.sigmoid((u_rep * item_emb).sum(dim=-1))
        return scores

    def score_all_items(self, u_triples_for_user, n_items, device):
        """
        Tính điểm dự đoán cho TOÀN BỘ items trong danh mục đối với 1 user cụ thể.
        Sử dụng thuật toán ma trận Vectorized: O(1) phép nhân ma trận thay vì vòng lặp for.
        """
        all_item_ids = torch.arange(n_items, device=device)
        V = self.entity_emb(all_item_ids) # [n_items, dim]
        
        u_0 = self.entity_emb(u_triples_for_user[0][0].squeeze(0)).mean(dim=0) # [dim]
        total_logits = torch.matmul(V, u_0) # [n_items]
        
        for l in range(self.n_layer):
            h, r, t = u_triples_for_user[l]
            h_e = self.entity_emb(h.squeeze(0)) # [utss, dim]
            r_e = self.relation_emb(r.squeeze(0)) # [utss, dim]
            t_e = self.entity_emb(t.squeeze(0)) # [utss, dim]
            
            Rh = r_e * h_e # [utss, dim]
            logits = torch.matmul(V, Rh.T) / np.sqrt(self.dim) # [n_items, utss]
            P = F.softmax(logits, dim=-1) # [n_items, utss]
            O_l = torch.matmul(P, t_e) # [n_items, dim]
            total_logits = total_logits + (O_l * V).sum(dim=-1)
            
        return torch.sigmoid(total_logits).detach().cpu().numpy()


class RippleNetTrainer:
    """
    Trainer cho mô hình RippleNet với BCE Loss và Adam Optimizer.
    """
    def __init__(self, model, lr=0.002, weight_decay=1e-5, device="cuda"):
        self.model = model.to(device)
        self.device = device
        self.optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad, self.model.parameters()),
            lr=lr,
            weight_decay=weight_decay
        )
        self.criterion = nn.BCELoss()

    def _get_user_triples(self, user_ids, user_triple_set, n_layer, utss=16):
        user_triples = []
        for l in range(n_layer):
            h, r, t = [], [], []
            for u in user_ids:
                if u in user_triple_set and len(user_triple_set[u]) > l:
                    trip = user_triple_set[u][l]
                    h.append(trip[0])
                    r.append(trip[1])
                    t.append(trip[2])
                else:
                    h.append([0] * utss)
                    r.append([0] * utss)
                    t.append([0] * utss)
            user_triples.append((
                torch.LongTensor(h).to(self.device),
                torch.LongTensor(r).to(self.device),
                torch.LongTensor(t).to(self.device)
            ))
        return user_triples

    def train_epoch(self, train_data, user_triple_set, n_layer, batch_size=1024, utss=16):
        self.model.train()
        indices = np.arange(len(train_data))
        np.random.shuffle(indices)
        total_loss = 0.0
        
        for s in range(0, len(train_data), batch_size):
            b_idx = indices[s:s + batch_size]
            b = train_data[b_idx]
            users = b[:, 0]
            items = torch.LongTensor(b[:, 1]).to(self.device)
            labels = torch.FloatTensor(b[:, 2]).to(self.device)
            u_triples = self._get_user_triples(users, user_triple_set, n_layer, utss=utss)
            
            self.optimizer.zero_grad()
            preds = self.model(items, u_triples)
            loss = self.criterion(preds, labels)
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item()
            
        return total_loss / (len(train_data) // batch_size + 1)

    def evaluate(self, test_data, user_triple_set, n_layer, batch_size=2048, utss=16):
        self.model.eval()
        scores = []
        with torch.no_grad():
            for s in range(0, len(test_data), batch_size):
                b = test_data[s:s + batch_size]
                users = b[:, 0]
                items = torch.LongTensor(b[:, 1]).to(self.device)
                u_triples = self._get_user_triples(users, user_triple_set, n_layer, utss=utss)
                pred = self.model(items, u_triples)
                scores.append(pred.cpu().numpy())
        scores = np.concatenate(scores)
        labels = test_data[:, 2]
        auc = roc_auc_score(labels, scores)
        pred_bin = (scores >= 0.5).astype(int)
        f1 = f1_score(labels, pred_bin, zero_division=0)
        acc = accuracy_score(labels, pred_bin)
        return auc, f1, acc, scores


## PHẦN C: MÔ HÌNH CKAN (SIGIR 2020) — MẠNG CHÚ Ý ĐỒ THỊ TRI THỨC HỢP TÁC

Bài báo: **CKAN: Collaborative Knowledge-aware Attentive Network for Recommender Systems**  
Tác giả: *Ze Wang, Guangyan Lin, Huanhuan Tan, Qingquan Chen, Xihan Liu* (ACM SIGIR 2020)  
Tham khảo: [weberrr/CKAN](https://github.com/weberrr/CKAN)

---

### 1. Kiến trúc mạng hai nhánh (Dual-Branch Ripple Propagation)
CKAN xây dựng cơ chế lan truyền hai chiều đồng thời:
1. **User Ripple Propagation**: Bắt đầu từ tập các item trong lịch sử người dùng, lan truyền sang các thực thể và người dùng khác trong mạng cộng tác.
2. **Item Ripple Propagation**: Bắt đầu từ item ứng viên $v$, mở rộng ngữ nghĩa sang các thực thể láng giềng liên kết trên Knowledge Graph.
3. **Knowledge-Aware Attention Layer (MLP 3 tầng)**:  
   Mạng nơ-ron đa tầng MLP tính trọng số chú ý phi tuyến giữa thực thể đầu và quan hệ:
   $$s(h, r) = \mathbf{W}_3 \cdot \text{ReLU}\left( \mathbf{W}_2 \cdot \text{ReLU}\left( \mathbf{W}_1 [\mathbf{h}; \mathbf{r}] \right) \right)$$
   $$\alpha(h, r) = \text{Softmax}\left( \sigma(s(h, r)) \right)$$
   $$\mathbf{e}^l = \sum_{(h_i, r_i, t_i)} \alpha(h_i, r_i) \mathbf{t}_i$$

4. **Bộ tổng hợp nối tiếp (Concat Aggregator)**:
   Vector biểu diễn cuối cùng của User và Item được tổng hợp từ toàn bộ các tầng lan truyền:
   $$\mathbf{e}_u = [\mathbf{e}_u^L; \dots; \mathbf{e}_u^1; \mathbf{e}_u^0], \qquad \mathbf{e}_v = [\mathbf{e}_v^L; \dots; \mathbf{e}_v^1; \mathbf{e}_v^0]$$
   $$\hat{y}_{uv} = \sigma\left( \mathbf{e}_u^T \mathbf{e}_v \right)$$


In [ ]:
# ============================================================
# Module Attention của mô hình CKAN
# ============================================================
class KnowledgeAwareAttentionLayer(nn.Module):
    """
    Lớp Attention MLP 3 tầng (Linear -> ReLU -> Linear -> ReLU -> Linear -> Sigmoid -> Softmax).
    Tính trọng số chú ý giữa thực thể đầu h và quan hệ r để tổng hợp thực thể đuôi t.
    """
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.attention = nn.Sequential(
            nn.Linear(dim * 2, dim, bias=False),
            nn.ReLU(),
            nn.Linear(dim, dim, bias=False),
            nn.ReLU(),
            nn.Linear(dim, 1, bias=False),
            nn.Sigmoid(),
        )
        for layer in self.attention:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)

    def forward(self, h_emb, r_emb, t_emb):
        # [batch_size, triple_set_size]
        att_weights = self.attention(torch.cat((h_emb, r_emb), dim=-1)).squeeze(-1)
        att_weights_norm = F.softmax(att_weights, dim=-1)
        # [batch_size, triple_set_size, dim] -> [batch_size, dim]
        emb_i = torch.mul(att_weights_norm.unsqueeze(-1), t_emb).sum(dim=1)
        return emb_i, att_weights_norm


In [ ]:
class CKAN(nn.Module):
    """
    Mô hình CKAN (SIGIR 2020):
    - Dual-branch: User Ripple Sets & Item Ripple Sets.
    - Knowledge-Aware Attention Layer trên từng tầng láng giềng.
    - Aggregator: Concat hoặc Sum vector biểu diễn các tầng.
    """
    def __init__(self, n_entity, n_relation, dim=64, n_layer=1, agg="concat"):
        super().__init__()
        self.n_entity = n_entity
        self.n_relation = n_relation
        self.dim = dim
        self.n_layer = n_layer
        self.agg = agg
        
        self.entity_emb = nn.Embedding(n_entity, dim)
        self.relation_emb = nn.Embedding(n_relation, dim)
        nn.init.xavier_uniform_(self.entity_emb.weight)
        nn.init.xavier_uniform_(self.relation_emb.weight)
        
        self.attention_layer = KnowledgeAwareAttentionLayer(dim)

    def _knowledge_attention(self, h_emb, r_emb, t_emb):
        emb_i, _ = self.attention_layer(h_emb, r_emb, t_emb)
        return emb_i

    def forward(self, items, user_triples, item_triples):
        user_embeddings = []
        user_emb_0 = self.entity_emb(user_triples[0][0]).mean(dim=1)
        user_embeddings.append(user_emb_0)
        
        for i in range(self.n_layer):
            h_emb = self.entity_emb(user_triples[i][0])
            r_emb = self.relation_emb(user_triples[i][1])
            t_emb = self.entity_emb(user_triples[i][2])
            user_embeddings.append(self._knowledge_attention(h_emb, r_emb, t_emb))

        item_embeddings = []
        item_emb_0 = self.entity_emb(items)
        item_embeddings.append(item_emb_0)
        
        for i in range(self.n_layer):
            h_emb = self.entity_emb(item_triples[i][0])
            r_emb = self.relation_emb(item_triples[i][1])
            t_emb = self.entity_emb(item_triples[i][2])
            item_embeddings.append(self._knowledge_attention(h_emb, r_emb, t_emb))

        if self.agg == "concat":
            e_u = torch.cat(user_embeddings, dim=-1)
            e_v = torch.cat(item_embeddings, dim=-1)
        elif self.agg == "sum":
            e_u = torch.stack(user_embeddings, dim=0).sum(dim=0)
            e_v = torch.stack(item_embeddings, dim=0).sum(dim=0)
        else:
            e_u = user_embeddings[-1]
            e_v = item_embeddings[-1]
            
        scores = (e_u * e_v).sum(dim=1)
        return torch.sigmoid(scores)

    def get_item_embeddings(self, item_ids, item_triple_set, device):
        with torch.no_grad():
            items_tensor = torch.LongTensor(item_ids).to(device)
            item_embeddings = [self.entity_emb(items_tensor)]
            
            for l in range(self.n_layer):
                h, r, t = [], [], []
                for it in item_ids:
                    trip = item_triple_set[it][l]
                    h.append(trip[0])
                    r.append(trip[1])
                    t.append(trip[2])
                h_emb = self.entity_emb(torch.LongTensor(h).to(device))
                r_emb = self.relation_emb(torch.LongTensor(r).to(device))
                t_emb = self.entity_emb(torch.LongTensor(t).to(device))
                item_embeddings.append(self._knowledge_attention(h_emb, r_emb, t_emb))
                
            if self.agg == "concat":
                return torch.cat(item_embeddings, dim=-1)
            elif self.agg == "sum":
                return torch.stack(item_embeddings, dim=0).sum(dim=0)
            else:
                return item_embeddings[-1]

    def get_user_embeddings(self, user_triples):
        with torch.no_grad():
            user_embeddings = [self.entity_emb(user_triples[0][0]).mean(dim=1)]
            for i in range(self.n_layer):
                h_emb = self.entity_emb(user_triples[i][0])
                r_emb = self.relation_emb(user_triples[i][1])
                t_emb = self.entity_emb(user_triples[i][2])
                user_embeddings.append(self._knowledge_attention(h_emb, r_emb, t_emb))
            if self.agg == "concat":
                return torch.cat(user_embeddings, dim=-1)
            elif self.agg == "sum":
                return torch.stack(user_embeddings, dim=0).sum(dim=0)
            else:
                return user_embeddings[-1]


In [ ]:
class CKANTrainer:
    """
    Trainer cho mô hình CKAN: Huấn luyện mini-batch với Adam Optimizer và BCE Loss.
    """
    def __init__(self, model, lr=0.002, weight_decay=1e-5, device="cuda"):
        self.model = model.to(device)
        self.device = device
        self.optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad, self.model.parameters()),
            lr=lr,
            weight_decay=weight_decay
        )
        self.criterion = nn.BCELoss()

    def _get_triples(self, ids, triples_dict, l, default_size=32):
        h, r, t = [], [], []
        for idx in ids:
            if idx in triples_dict and len(triples_dict[idx]) > l:
                trip = triples_dict[idx][l]
                h.append(trip[0])
                r.append(trip[1])
                t.append(trip[2])
            else:
                h.append([0] * default_size)
                r.append([0] * default_size)
                t.append([0] * default_size)
        return (
            torch.LongTensor(h).to(self.device),
            torch.LongTensor(r).to(self.device),
            torch.LongTensor(t).to(self.device)
        )

    def train_epoch(self, train_data, user_triple_set, item_triple_set, n_layer, batch_size=2048):
        self.model.train()
        indices = np.arange(len(train_data))
        np.random.shuffle(indices)
        total_loss = 0.0
        
        for s in range(0, len(train_data), batch_size):
            b_idx = indices[s:s + batch_size]
            b = train_data[b_idx]
            
            users = b[:, 0]
            items = torch.LongTensor(b[:, 1]).to(self.device)
            labels = torch.FloatTensor(b[:, 2]).to(self.device)
            
            u_tr = [self._get_triples(users, user_triple_set, l, default_size=len(user_triple_set[list(user_triple_set.keys())[0]][0][0])) for l in range(n_layer)]
            i_tr = [self._get_triples(b[:, 1], item_triple_set, l, default_size=len(item_triple_set[0][0][0])) for l in range(n_layer)]
            
            self.optimizer.zero_grad()
            preds = self.model(items, u_tr, i_tr)
            loss = self.criterion(preds, labels)
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item()
            
        return total_loss / (len(train_data) // batch_size + 1)

    def evaluate(self, test_data, user_triple_set, item_triple_set, n_layer, batch_size=4096):
        self.model.eval()
        scores = []
        with torch.no_grad():
            for s in range(0, len(test_data), batch_size):
                b = test_data[s:s + batch_size]
                users = b[:, 0]
                items = torch.LongTensor(b[:, 1]).to(self.device)
                
                u_tr = [self._get_triples(users, user_triple_set, l, default_size=len(user_triple_set[list(user_triple_set.keys())[0]][0][0])) for l in range(n_layer)]
                i_tr = [self._get_triples(b[:, 1], item_triple_set, l, default_size=len(item_triple_set[0][0][0])) for l in range(n_layer)]
                
                preds = self.model(items, u_tr, i_tr)
                scores.append(preds.cpu().numpy())
                
        scores = np.concatenate(scores)
        labels = test_data[:, 2]
        auc = roc_auc_score(labels, scores)
        pred_bin = (scores >= 0.5).astype(int)
        f1 = f1_score(labels, pred_bin, zero_division=0)
        acc = accuracy_score(labels, pred_bin)
        return auc, f1, acc, scores


In [ ]:
# ============================================================
# Đánh giá xếp hạng Top-K (Recall@K)
# ============================================================
class TopKRecommenderEvaluator:
    """
    Đánh giá xếp hạng Top-K (Top-K Recommendation Evaluation) chuẩn hóa theo paper:
    - Thang đo chuẩn: Recall@K với K in {5, 10, 20, 50, 100}
    - Đánh giá trên 100 người dùng mẫu có tương tác positive trong tập test
    - Loại bỏ các item người dùng đã tương tác trong tập train trước khi xếp hạng
    """
    def __init__(self, k_list=[5, 10, 20, 50, 100]):
        self.k_list = k_list
        self.max_k = max(k_list)

    def evaluate_model(self, score_func, eval_users, train_pos_dict, test_pos_dict, n_items):
        metrics = {f"Recall@{k}": [] for k in self.k_list}
        
        for u in eval_users:
            test_pos = test_pos_dict.get(u, set())
            if not test_pos:
                continue
            
            # Tính điểm số cho toàn bộ items trong danh mục
            scores = score_func(u)
            if not isinstance(scores, np.ndarray):
                scores = np.array(scores)
            else:
                scores = scores.copy()
                
            # Loại trừ item đã tương tác trong tập train
            train_pos = train_pos_dict.get(u, set())
            if train_pos:
                scores[list(train_pos)] = -1e9
                
            # Lấy Top max_k item có điểm số cao nhất
            top_items = np.argpartition(scores, -self.max_k)[-self.max_k:]
            top_items = top_items[np.argsort(-scores[top_items])]
            
            num_pos = len(test_pos)
            for k in self.k_list:
                top_k = set(top_items[:k])
                hits = len(top_k & test_pos)
                metrics[f"Recall@{k}"].append(hits / num_pos)
                
        return {m: float(np.mean(vals)) if vals else 0.0 for m, vals in metrics.items()}


## PHẦN D: THỰC NGHIỆM ĐÁNH GIÁ (MOVIE, BOOK, MUSIC)

Quy trình thực nghiệm:
1. **Phân chia dữ liệu theo tỷ lệ 6:2:2**:
   - **Train set (60%)**: Cập nhật trọng số mô hình.
   - **Validation set (20%)**: Theo dõi quá trình hội tụ và lưu checkpoint có `val_auc` cao nhất.
   - **Test set (20%)**: Đánh giá độc lập chất lượng mô hình.
2. **Huấn luyện 4 mô hình**:
   - `MostPopular`: Heuristic dựa trên tần suất positive trong train set.
   - `Matrix Factorization`: Lọc cộng tác ma trận, chọn checkpoint theo `val_auc`.
   - `RippleNet`: Lan truyền gợn sóng 1 chiều, chọn checkpoint theo `val_auc`.
   - `CKAN`: Lan truyền 2 chiều kết hợp MLP Attention, chọn checkpoint theo `val_auc`.
3. **Đo lường các chỉ số**:
   - CTR: `ROC-AUC`, `F1-Score`, `Accuracy`.
   - Top-K: `Recall@5, @10, @20, @50, @100` trên 100 người dùng mẫu.
   - Độ thưa (Sparsity): Đo lường ROC-AUC qua 6 tỷ lệ train ($10\%, 20\%, 40\%, 60\%, 80\%, 100\%$).


In [ ]:
# ============================================================
# THỰC NGHIỆM BENCHMARK TOÀN DIỆN (6:2:2 Split & Validation-guided Checkpointing)
# ============================================================
import copy

def evaluate_predictions(labels, scores):
    auc = roc_auc_score(labels, scores)
    pred_bin = (scores >= 0.5).astype(int)
    f1 = f1_score(labels, pred_bin, zero_division=0)
    acc = accuracy_score(labels, pred_bin)
    return auc, f1, acc

def run_comprehensive_benchmark(ds_name):
    print(f"\n--- DATASET: {ds_name.upper()} ---")
    
    cfg = CONFIG[ds_name]
    rating_np = np.load(f"./data/{ds_name}/ratings_final.npy")
    kg_np = np.load(f"./data/{ds_name}/kg_final.npy")
    
    n_user = int(rating_np[:, 0].max()) + 1
    n_item = max(int(rating_np[:, 1].max()), int(kg_np[:, 0].max())) + 1
    n_entity = max(int(rating_np[:, 1].max()), int(kg_np[:, 0].max()), int(kg_np[:, 2].max())) + 1
    n_relation = int(kg_np[:, 1].max()) + 1
    
    print(f"Users: {n_user:,} | Items: {n_item:,} | Ratings: {len(rating_np):,} | Triples: {len(kg_np):,}")
    
    # ------------------------------------------------------------
    # PHÂN CHIA DỮ LIỆU TỶ LỆ 6:2:2 (60% Train / 20% Val / 20% Test)
    # ------------------------------------------------------------
    np.random.seed(42)
    shuffled_ratings = rating_np.copy()
    np.random.shuffle(shuffled_ratings)
    n_samples = len(shuffled_ratings)
    train_end = int(n_samples * 0.6)
    eval_end = int(n_samples * 0.8)
    
    train_data = shuffled_ratings[:train_end]
    eval_data = shuffled_ratings[train_end:eval_end]
    test_data = shuffled_ratings[eval_end:]
    
    # 1. Trích xuất tương tác positive trong train
    train_pos_dict = defaultdict(set)
    for u, i, r in train_data:
        if r == 1:
            train_pos_dict[u].add(i)
        
    # Chuẩn hóa theo repo gốc weberrr/CKAN: lọc user có ít nhất 1 positive rating trong train
    valid_train_users = np.array(list(train_pos_dict.keys()))
    train_data = train_data[np.isin(train_data[:, 0], valid_train_users)]
    eval_data = eval_data[np.isin(eval_data[:, 0], valid_train_users)]
    test_data = test_data[np.isin(test_data[:, 0], valid_train_users)]
    
    test_pos_dict = defaultdict(set)
    for u, i, r in test_data:
        if r == 1:
            test_pos_dict[u].add(i)
        
    print(f"Phân chia: Train={len(train_data):,} | Eval={len(eval_data):,} | Test={len(test_data):,}")
    
    # Lấy mẫu 100 users để đánh giá Top-K Recall
    eval_users = [u for u in test_pos_dict if len(test_pos_dict[u]) >= 3][:100]
    topk_evaluator = TopKRecommenderEvaluator(k_list=[5, 10, 20, 50, 100])
    bs = cfg["batch_size"]
    crit = nn.BCELoss()
    
    # ------------------------------------------------------------
    # 1. MOST POPULAR
    # ------------------------------------------------------------
    pop = MostPopularBaseline()
    pop.fit(train_data, n_item)
    pop_scores = pop.predict(test_data[:, 0], test_data[:, 1])
    pop_auc, pop_f1, pop_acc = evaluate_predictions(test_data[:, 2], pop_scores)
    pop_topk = topk_evaluator.evaluate_model(lambda u: pop.score_all_items(u, n_item), eval_users, train_pos_dict, test_pos_dict, n_item)
    print(f"  [1/4] MostPopular       : Test_AUC={pop_auc:.4f} | F1={pop_f1:.4f} | ACC={pop_acc:.4f} | Rec@10={pop_topk['Recall@10']:.4f} | Rec@50={pop_topk['Recall@50']:.4f}")
    
    # ------------------------------------------------------------
    # 2. MATRIX FACTORIZATION (Chọn epoch tốt nhất theo Validation)
    # ------------------------------------------------------------
    mf = MatrixFactorizationBaseline(n_user, n_item, dim=cfg["dim"]).to(device)
    opt_mf = torch.optim.Adam(mf.parameters(), lr=cfg["lr"], weight_decay=1e-5)
    
    best_mf_val_auc = -1.0
    best_mf_state = None
    
    for ep in range(cfg["n_epochs"]):
        mf.train()
        for s in range(0, len(train_data), bs):
            b = train_data[s:s+bs]
            opt_mf.zero_grad()
            pred = mf(torch.LongTensor(b[:, 0]).to(device), torch.LongTensor(b[:, 1]).to(device))
            loss = crit(pred, torch.FloatTensor(b[:, 2]).to(device))
            loss.backward()
            opt_mf.step()
            
        # Theo dõi trên Validation Set
        mf.eval()
        with torch.no_grad():
            v_sc = [mf(torch.LongTensor(b[:, 0]).to(device), torch.LongTensor(b[:, 1]).to(device)).cpu().numpy() for b in [eval_data[s:s+bs] for s in range(0, len(eval_data), bs)]]
            v_sc = np.concatenate(v_sc)
        v_auc, _, _ = evaluate_predictions(eval_data[:, 2], v_sc)
        if v_auc > best_mf_val_auc:
            best_mf_val_auc = v_auc
            best_mf_state = copy.deepcopy(mf.state_dict())
            
    # Nạp Best Checkpoint để đánh giá Test Set
    if best_mf_state is not None:
        mf.load_state_dict(best_mf_state)
    mf.eval()
    with torch.no_grad():
        mf_scores = [mf(torch.LongTensor(b[:, 0]).to(device), torch.LongTensor(b[:, 1]).to(device)).cpu().numpy() for b in [test_data[s:s+bs] for s in range(0, len(test_data), bs)]]
        mf_scores = np.concatenate(mf_scores)
    mf_auc, mf_f1, mf_acc = evaluate_predictions(test_data[:, 2], mf_scores)
    mf_topk = topk_evaluator.evaluate_model(lambda u: mf.score_all_items(u, device), eval_users, train_pos_dict, test_pos_dict, n_item)
    print(f"  [2/4] MF                : Val_AUC={best_mf_val_auc:.4f} | Test_AUC={mf_auc:.4f} | F1={mf_f1:.4f} | ACC={mf_acc:.4f} | Rec@10={mf_topk['Recall@10']:.4f} | Rec@50={mf_topk['Recall@50']:.4f}")
    
    # ------------------------------------------------------------
    # CHUẨN BỊ RIPPLE SETS
    # ------------------------------------------------------------
    sampler = KnowledgeRippleSampler(kg_np, n_layer=cfg["n_layer"], itss=cfg["itss"], utss=cfg["utss"])
    item_triple_set = sampler.build_item_ripple_set(n_item)
    user_triple_set = sampler.build_user_ripple_set(train_pos_dict, n_users=n_user)
    
    # ------------------------------------------------------------
    # 3. RIPPLENET (Chọn checkpoint theo Validation)
    # ------------------------------------------------------------
    ripplenet = RippleNet(n_entity, n_relation, dim=cfg["dim"], n_layer=cfg["n_layer"]).to(device)
    ripple_trainer = RippleNetTrainer(ripplenet, lr=cfg["lr"], weight_decay=cfg["l2_weight"], device=device)
    
    best_rip_val_auc = -1.0
    best_rip_state = None
    
    for ep in range(cfg["n_epochs"]):
        ripple_trainer.train_epoch(train_data, user_triple_set, cfg["n_layer"], batch_size=bs, utss=cfg["utss"])
        v_auc, _, _, _ = ripple_trainer.evaluate(eval_data, user_triple_set, cfg["n_layer"], batch_size=bs, utss=cfg["utss"])
        if v_auc > best_rip_val_auc:
            best_rip_val_auc = v_auc
            best_rip_state = copy.deepcopy(ripplenet.state_dict())
            
    if best_rip_state is not None:
        ripplenet.load_state_dict(best_rip_state)
        
    ripple_auc, ripple_f1, ripple_acc, _ = ripple_trainer.evaluate(test_data, user_triple_set, cfg["n_layer"], batch_size=bs, utss=cfg["utss"])
    
    def ripple_score_fn(u):
        with torch.no_grad():
            u_trips = ripple_trainer._get_user_triples([u], user_triple_set, cfg["n_layer"], utss=cfg["utss"])
            return ripplenet.score_all_items(u_trips, n_item, device)
            
    ripple_topk = topk_evaluator.evaluate_model(ripple_score_fn, eval_users, train_pos_dict, test_pos_dict, n_item)
    print(f"  [3/4] RippleNet         : Val_AUC={best_rip_val_auc:.4f} | Test_AUC={ripple_auc:.4f} | F1={ripple_f1:.4f} | ACC={ripple_acc:.4f} | Rec@10={ripple_topk['Recall@10']:.4f} | Rec@50={ripple_topk['Recall@50']:.4f}")
    
    # ------------------------------------------------------------
    # 4. CKAN (Chọn checkpoint theo Validation)
    # ------------------------------------------------------------
    ckan = CKAN(n_entity, n_relation, dim=cfg["dim"], n_layer=cfg["n_layer"], agg=cfg["agg"]).to(device)
    ckan_trainer = CKANTrainer(ckan, lr=cfg["lr"], weight_decay=cfg["l2_weight"], device=device)
    
    best_ckan_val_auc = -1.0
    best_ckan_state = None
    
    for ep in range(cfg["n_epochs"]):
        ckan_trainer.train_epoch(train_data, user_triple_set, item_triple_set, cfg["n_layer"], batch_size=bs)
        v_auc, _, _, _ = ckan_trainer.evaluate(eval_data, user_triple_set, item_triple_set, cfg["n_layer"], batch_size=bs)
        if v_auc > best_ckan_val_auc:
            best_ckan_val_auc = v_auc
            best_ckan_state = copy.deepcopy(ckan.state_dict())
            
    if best_ckan_state is not None:
        ckan.load_state_dict(best_ckan_state)
        
    ckan_auc, ckan_f1, ckan_acc, _ = ckan_trainer.evaluate(test_data, user_triple_set, item_triple_set, cfg["n_layer"], batch_size=bs)
    
    # Tính ma trận item embeddings trên GPU từ Checkpoint tốt nhất
    all_item_matrix = []
    chunk_size = 2048
    for ch_s in range(0, n_item, chunk_size):
        ch_ids = list(range(ch_s, min(ch_s + chunk_size, n_item)))
        all_item_matrix.append(ckan.get_item_embeddings(ch_ids, item_triple_set, device))
    all_item_matrix = torch.cat(all_item_matrix, dim=0)
    
    def ckan_score_fn(u):
        with torch.no_grad():
            u_tr = []
            for l in range(cfg["n_layer"]):
                if u in user_triple_set and len(user_triple_set[u]) > l:
                    trip = user_triple_set[u][l]
                    uh = torch.LongTensor([trip[0]]).to(device)
                    ur = torch.LongTensor([trip[1]]).to(device)
                    ut = torch.LongTensor([trip[2]]).to(device)
                else:
                    uh = torch.LongTensor([[0] * cfg["utss"]]).to(device)
                    ur = torch.LongTensor([[0] * cfg["utss"]]).to(device)
                    ut = torch.LongTensor([[0] * cfg["utss"]]).to(device)
                u_tr.append((uh, ur, ut))
            u_emb = ckan.get_user_embeddings(u_tr)
            scores = torch.matmul(u_emb, all_item_matrix.T).squeeze(0)
            return scores.cpu().numpy()
            
    ckan_topk = topk_evaluator.evaluate_model(ckan_score_fn, eval_users, train_pos_dict, test_pos_dict, n_item)
    print(f"  [4/4] CKAN              : Val_AUC={best_ckan_val_auc:.4f} | Test_AUC={ckan_auc:.4f} | F1={ckan_f1:.4f} | ACC={ckan_acc:.4f} | Rec@10={ckan_topk['Recall@10']:.4f} | Rec@50={ckan_topk['Recall@50']:.4f}")

    # ------------------------------------------------------------
    # TỰ ĐỘNG LƯU MODEL VÀ TẤT CẢ ARTIFACTS
    # ------------------------------------------------------------
    save_dir = f"./saved_models/{ds_name}"
    os.makedirs(save_dir, exist_ok=True)
    torch.save(ckan.state_dict(), os.path.join(save_dir, "ckan_model.pt"))
    np.save(os.path.join(save_dir, "item_embeddings.npy"), all_item_matrix.cpu().numpy())
    model_config = {
        "dataset": ds_name,
        "n_user": int(n_user), "n_item": int(n_item),
        "n_entity": int(n_entity), "n_relation": int(n_relation),
        "dim": int(cfg["dim"]), "n_layer": int(cfg["n_layer"]),
        "itss": int(cfg["itss"]), "utss": int(cfg["utss"]),
        "agg": str(cfg["agg"])
    }
    with open(os.path.join(save_dir, "config.json"), "w", encoding="utf-8") as f:
        json.dump(model_config, f, indent=2)
    with open(os.path.join(save_dir, "item_triple_set.pkl"), "wb") as f:
        pickle.dump(dict(item_triple_set), f)
    with open(os.path.join(save_dir, "user_history.pkl"), "wb") as f:
        pickle.dump(dict(train_pos_dict), f)
    map_file = f"./data/{ds_name}/item_index2entity_id.txt"
    if os.path.exists(map_file):
        shutil.copy(map_file, os.path.join(save_dir, "item_index2entity_id.txt"))
    # Saved checkpoint

    # ------------------------------------------------------------
    # THỰC NGHIỆM ĐỘ THƯA (SPARSITY: 6 MỐC VỚI VALIDATION TRACKING & ADAPTIVE BATCH)
    # Ratios = [0.1, 0.2, 0.4, 0.6, 0.8, 1.0] đo bằng ROC-AUC
    # ------------------------------------------------------------
    print("  Sparsity:")
    sparsity_ratios = [0.1, 0.2, 0.4, 0.6, 0.8, 1.0]
    mf_sparsity_aucs = []
    ripple_sparsity_aucs = []
    ckan_sparsity_aucs = []
    sp_epochs = cfg["n_epochs"]
    
    for r in sparsity_ratios:
        sub_len = max(int(len(train_data) * r), 128)
        sub_tr = train_data[:sub_len]
        # Co giãn batch size thích ứng để đảm bảo đủ số bước gradient trên tập con nhỏ
        sub_bs = min(bs, max(128, len(sub_tr) // 8))
        
        # 1. MF Sparsity
        m_sp = MatrixFactorizationBaseline(n_user, n_item, dim=cfg["dim"]).to(device)
        opt_s = torch.optim.Adam(m_sp.parameters(), lr=cfg["lr"], weight_decay=1e-5)
        best_m_v_auc, best_m_st = -1.0, None
        for _ in range(sp_epochs):
            m_sp.train()
            for s in range(0, len(sub_tr), sub_bs):
                b = sub_tr[s:s+sub_bs]
                opt_s.zero_grad()
                crit(m_sp(torch.LongTensor(b[:, 0]).to(device), torch.LongTensor(b[:, 1]).to(device)), torch.FloatTensor(b[:, 2]).to(device)).backward()
                opt_s.step()
            m_sp.eval()
            with torch.no_grad():
                v_sc = [m_sp(torch.LongTensor(b[:, 0]).to(device), torch.LongTensor(b[:, 1]).to(device)).cpu().numpy() for b in [eval_data[s:s+bs] for s in range(0, len(eval_data), bs)]]
                v_sc = np.concatenate(v_sc)
            v_auc, _, _ = evaluate_predictions(eval_data[:, 2], v_sc)
            if v_auc > best_m_v_auc:
                best_m_v_auc = v_auc
                best_m_st = copy.deepcopy(m_sp.state_dict())
        if best_m_st is not None:
            m_sp.load_state_dict(best_m_st)
        m_sp.eval()
        with torch.no_grad():
            sc = [m_sp(torch.LongTensor(b[:, 0]).to(device), torch.LongTensor(b[:, 1]).to(device)).cpu().numpy() for b in [test_data[s:s+bs] for s in range(0, len(test_data), bs)]]
            sc = np.concatenate(sc)
        mf_auc_r, _, _ = evaluate_predictions(test_data[:, 2], sc)
        mf_sparsity_aucs.append(mf_auc_r)
        
        # 2. RippleNet Sparsity
        rip_sp = RippleNet(n_entity, n_relation, dim=cfg["dim"], n_layer=cfg["n_layer"]).to(device)
        rip_tr_sp = RippleNetTrainer(rip_sp, lr=cfg["lr"], weight_decay=cfg["l2_weight"], device=device)
        best_r_v_auc, best_r_st = -1.0, None
        for _ in range(sp_epochs):
            rip_tr_sp.train_epoch(sub_tr, user_triple_set, cfg["n_layer"], batch_size=sub_bs, utss=cfg["utss"])
            v_auc, _, _, _ = rip_tr_sp.evaluate(eval_data, user_triple_set, cfg["n_layer"], batch_size=bs, utss=cfg["utss"])
            if v_auc > best_r_v_auc:
                best_r_v_auc = v_auc
                best_r_st = copy.deepcopy(rip_sp.state_dict())
        if best_r_st is not None:
            rip_sp.load_state_dict(best_r_st)
        rip_auc_r, _, _, _ = rip_tr_sp.evaluate(test_data, user_triple_set, cfg["n_layer"], batch_size=bs, utss=cfg["utss"])
        ripple_sparsity_aucs.append(rip_auc_r)
        
        # 3. CKAN Sparsity
        ck_sp = CKAN(n_entity, n_relation, dim=cfg["dim"], n_layer=cfg["n_layer"], agg=cfg["agg"])
        ck_tr_sp = CKANTrainer(ck_sp, lr=cfg["lr"], weight_decay=cfg["l2_weight"], device=device)
        best_c_v_auc, best_c_st = -1.0, None
        for _ in range(sp_epochs):
            ck_tr_sp.train_epoch(sub_tr, user_triple_set, item_triple_set, cfg["n_layer"], batch_size=sub_bs)
            v_auc, _, _, _ = ck_tr_sp.evaluate(eval_data, user_triple_set, item_triple_set, cfg["n_layer"], batch_size=bs)
            if v_auc > best_c_v_auc:
                best_c_v_auc = v_auc
                best_c_st = copy.deepcopy(ck_sp.state_dict())
        if best_c_st is not None:
            ck_sp.load_state_dict(best_c_st)
        ck_auc_r, _, _, _ = ck_tr_sp.evaluate(test_data, user_triple_set, item_triple_set, cfg["n_layer"], batch_size=bs)
        ckan_sparsity_aucs.append(ck_auc_r)
        diff_r = (ck_auc_r - mf_auc_r) * 100
        print(f"    - Mốc {int(r*100):3d}% Train: MF={mf_auc_r:.4f} | RippleNet={rip_auc_r:.4f} | CKAN={ck_auc_r:.4f} ")
        
    # End sparsity loop

    return {
        "summary": {
            "Dataset": ds_name.capitalize(),
            "Users": n_user, "Items": n_item, "Ratings": len(rating_np), "KG_Triples": len(kg_np),
            "MostPop_AUC": pop_auc, "MF_AUC": mf_auc, "Ripple_AUC": ripple_auc, "CKAN_AUC": ckan_auc,
            "MostPop_F1": pop_f1, "MF_F1": mf_f1, "Ripple_F1": ripple_f1, "CKAN_F1": ckan_f1,
            "MostPop_ACC": pop_acc, "MF_ACC": mf_acc, "Ripple_ACC": ripple_acc, "CKAN_ACC": ckan_acc,
            "MF_Rec10": mf_topk['Recall@10'], "Ripple_Rec10": ripple_topk['Recall@10'], "CKAN_Rec10": ckan_topk['Recall@10'],
            "MF_Rec50": mf_topk['Recall@50'], "Ripple_Rec50": ripple_topk['Recall@50'], "CKAN_Rec50": ckan_topk['Recall@50'],
            "MF_Sparse10_AUC": mf_sparsity_aucs[0], "Ripple_Sparse10_AUC": ripple_sparsity_aucs[0], "CKAN_Sparse10_AUC": ckan_sparsity_aucs[0]
        },
        "topk": {
            "MostPop": pop_topk,
            "MF": mf_topk,
            "RippleNet": ripple_topk,
            "CKAN": ckan_topk
        },
        "sparsity": {
            "ratios": sparsity_ratios,
            "MF": mf_sparsity_aucs,
            "RippleNet": ripple_sparsity_aucs,
            "CKAN": ckan_sparsity_aucs
        }
    }

# Chạy thực nghiệm trên cả 3 tập dữ liệu
benchmark_results = {}
for ds in active_datasets:
    benchmark_results[ds] = run_comprehensive_benchmark(ds)

# Tổng hợp bảng so sánh kết quả
df_results = pd.DataFrame([benchmark_results[ds]["summary"] for ds in active_datasets])
print("\n" + "="*80)
print("TỔNG HỢP HIỆU SUẤT CTR VÀ ĐỘ THỬA (3 DATASETS)")
print("="*80)
display_cols = ["Dataset", "Ratings", "MF_AUC", "Ripple_AUC", "CKAN_AUC", "MF_F1", "Ripple_F1", "CKAN_F1", "MF_Sparse10_AUC", "CKAN_Sparse10_AUC"]
print(df_results[display_cols].to_string(index=False))


In [ ]:
# ============================================================
# BẢNG TỔNG HỢP KẾT QUẢ CHI TIẾT (CTR, Top-K & 6 Mốc Sparsity)
# Trình bày trực quan, chuẩn xác để theo dõi và đưa vào báo cáo
# ============================================================
from IPython.display import display

print("="*90)
print("BẢNG 1: HIỆU SUẤT DỰ ĐOÁN CTR (ROC-AUC, F1-SCORE, ACCURACY)")
print("="*90)
ctr_rows = []
for ds in active_datasets:
    s = benchmark_results[ds]["summary"]
    for m, label in [("MostPop", "MostPopular"), ("MF", "MF"), ("Ripple", "RippleNet"), ("CKAN", "CKAN")]:
        ctr_rows.append({
            "Dataset": ds.capitalize(),
            "Mô hình": label,
            "ROC-AUC": f"{s[f'{m}_AUC']:.4f}",
            "F1-Score": f"{s[f'{m}_F1']:.4f}",
            "Accuracy": f"{s[f'{m}_ACC']:.4f}"
        })
df_ctr = pd.DataFrame(ctr_rows)
print(df_ctr.to_string(index=False))
display(df_ctr)

print("\n" + "="*90)
print("BẢNG 2: HIỆU SUẤT XẾP HẠNG TOP-K (RECALL@K VỚI K in [5, 10, 20, 50, 100])")
print("="*90)
k_values = [5, 10, 20, 50, 100]
topk_rows = []
for ds in active_datasets:
    topk_data = benchmark_results[ds]["topk"]
    for m, label in [("MostPop", "MostPopular"), ("MF", "MF"), ("RippleNet", "RippleNet"), ("CKAN", "CKAN")]:
        row = {"Dataset": ds.capitalize(), "Mô hình": label}
        for k in k_values:
            row[f"Recall@{k}"] = f"{topk_data[m][f'Recall@{k}']:.4f}"
        topk_rows.append(row)
df_topk = pd.DataFrame(topk_rows)
print(df_topk.to_string(index=False))
display(df_topk)

print("\n" + "="*90)
print("BẢNG 3: KHẢ NĂNG CHỐNG CHỊU DỮ LIỆU THƯA (ROC-AUC QUA ĐẦY ĐỦ 6 MỐC DATA)")
print("="*90)
sp_rows = []
sparsity_ratios = [0.1, 0.2, 0.4, 0.6, 0.8, 1.0]
for ds in active_datasets:
    sp_data = benchmark_results[ds]["sparsity"]
    for m, label in [("MF", "MF"), ("RippleNet", "RippleNet"), ("CKAN", "CKAN")]:
        row = {"Dataset": ds.capitalize(), "Mô hình": label}
        for r, auc_val in zip(sparsity_ratios, sp_data[m]):
            row[f"{int(r*100)}% Data"] = f"{auc_val:.4f}"
        sp_rows.append(row)
df_sp = pd.DataFrame(sp_rows)
print(df_sp.to_string(index=False))
display(df_sp)


In [ ]:
# ============================================================
# TRỰC QUAN HÓA KẾT QUẢ THỰC NGHIỆM
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"font.sans-serif": "DejaVu Sans", "font.size": 10})

# ------------------------------------------------------------
# HÌNH 1: SO SÁNH HIỆU SUẤT CTR (ROC-AUC, F1-SCORE, ACCURACY)
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), dpi=300)
x = np.arange(len(df_results))
width = 0.20
models = ["MostPop", "MF", "Ripple", "CKAN"]
labels = ["MostPopular", "MF", "RippleNet", "CKAN"]
colors = ["#9CA3AF", "#3B82F6", "#8B5CF6", "#D97706"]

metrics = [("AUC", "ROC-AUC Score", (0.5, 1.05)),
           ("F1", "F1-Score", (0.0, 1.05)),
           ("ACC", "Accuracy", (0.5, 1.05))]

for idx, (m_key, m_name, y_lim) in enumerate(metrics):
    ax = axes[idx]
    for m_i, (m_code, m_label, col) in enumerate(zip(models, labels, colors)):
        col_name = f"{m_code}_{m_key}"
        vals = df_results[col_name]
        pos = x + (m_i - 1.5) * width
        bars = ax.bar(pos, vals, width, label=m_label, color=col, edgecolor="white", alpha=0.95)
        for b in bars:
            h = b.get_height()
            ax.text(b.get_x() + b.get_width()/2, h + 0.012, f"{h:.3f}",
                    ha="center", va="bottom", fontsize=7.5, rotation=35)
    ax.set_xticks(x)
    ax.set_xticklabels(df_results["Dataset"], fontsize=11, fontweight="bold")
    ax.set_title(f"CTR: {m_name}", fontsize=12, fontweight="bold")
    ax.set_ylim(y_lim)
    ax.grid(axis="y", linestyle=":", alpha=0.7)
    if idx == 0:
        ax.legend(loc="lower right", framealpha=0.9)

plt.suptitle("Hình 1: So sánh hiệu suất dự đoán CTR trên 3 tập dữ liệu", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# HÌNH 2: ĐƯỜNG CONG RECALL@K (K in [5, 10, 20, 50, 100])
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=300)
k_values = [5, 10, 20, 50, 100]
markers = {"MostPop": "o", "MF": "s", "RippleNet": "^", "CKAN": "D"}

for idx, ds in enumerate(active_datasets):
    ax = axes[idx]
    topk_data = benchmark_results[ds]["topk"]
    
    for m_label, col in zip(["MostPop", "MF", "RippleNet", "CKAN"], ["#9CA3AF", "#3B82F6", "#8B5CF6", "#D97706"]):
        y_vals = [topk_data[m_label][f"Recall@{k}"] for k in k_values]
        ax.plot(k_values, y_vals, marker=markers[m_label], linewidth=2.2, label=m_label, color=col)
        
    ax.set_title(f"{ds.capitalize()} Dataset", fontsize=12, fontweight="bold")
    ax.set_xlabel("K", fontsize=11, fontweight="bold")
    ax.set_ylabel("Recall@K", fontsize=11, fontweight="bold")
    ax.set_xticks(k_values)
    ax.grid(True, linestyle=":", alpha=0.7)
    ax.legend(loc="upper left")

plt.suptitle("Hình 2: Đường cong Recall@K trên 100 người dùng mẫu", fontsize=14, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# HÌNH 3: KHẢ NĂNG CHỐNG CHỊU DỮ LIỆU THƯA (SPARSITY ANALYSIS: 6 MỐC)
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=300)
ratios_pct = [f"{int(r*100)}%" for r in [0.1, 0.2, 0.4, 0.6, 0.8, 1.0]]

for idx, ds in enumerate(active_datasets):
    ax = axes[idx]
    sp_data = benchmark_results[ds]["sparsity"]
    ratios = sp_data["ratios"]
    
    ax.plot(ratios_pct, sp_data["MF"], marker="s", linewidth=2.2, linestyle="--", label="MF", color="#3B82F6")
    ax.plot(ratios_pct, sp_data["RippleNet"], marker="^", linewidth=2.2, linestyle="-.", label="RippleNet", color="#8B5CF6")
    ax.plot(ratios_pct, sp_data["CKAN"], marker="D", linewidth=2.5, linestyle="-", label="CKAN", color="#D97706")
    

                
    ax.set_title(f"{ds.capitalize()} (Sparsity Curves)", fontsize=12, fontweight="bold")
    ax.set_xlabel("Tỷ lệ dữ liệu huấn luyện (Training Ratio)", fontsize=11, fontweight="bold")
    ax.set_ylabel("ROC-AUC", fontsize=11, fontweight="bold")
    ax.set_ylim(0.45, 1.02)
    ax.grid(True, linestyle=":", alpha=0.7)
    ax.legend(loc="lower right")

plt.suptitle("Hình 3: Khả năng chống chịu dữ liệu thưa theo tỷ lệ huấn luyện", fontsize=14, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# LƯU TRỮ VÀ NÉN MÔ HÌNH THÀNH FILE ZIP
# ============================================================
import zipfile

zip_path = "./ckan_tri_dataset_models.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk("./saved_models"):
        for file in files:
            full_path = os.path.join(root, file)
            rel_path = os.path.relpath(full_path, "./")
            zipf.write(full_path, rel_path)

zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f"Đã lưu mô hình và nén thành công: {zip_path} ({zip_size_mb:.2f} MB)")
